# Data Cleaning – Bank Marketing (`bank-full.csv`)

Notebook làm sạch dữ liệu chiến dịch marketing qua điện thoại của ngân hàng Bồ Đào Nha (biến mục tiêu `y` (deposit): khách có đăng ký tiền gửi có kỳ hạn hay không).


## 1. Thư viện và hằng số
Khai báo thư viện, danh sách tháng và tập nhãn hợp lệ của các biến phân loại (theo mô tả dataset).

In [1]:
import pandas as pd
import numpy as np

MONTHS = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
MONTH_NUM = {m: i + 1 for i, m in enumerate(MONTHS)}
DAYS_IN_MONTH = {1:31, 2:29, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}  # cho phép 29/02

ALLOWED = {
    'job': ['admin.','unknown','unemployed','management','housemaid','entrepreneur',
            'student','blue-collar','self-employed','retired','technician','services'],
    'marital':   ['married','divorced','single'],
    'education': ['unknown','secondary','primary','tertiary'],
    'default':   ['yes','no'],
    'housing':   ['yes','no'],
    'loan':      ['yes','no'],
    'contact':   ['unknown','telephone','cellular'],
    'month':     MONTHS,
    'poutcome':  ['unknown','other','failure','success'],
    'y':         ['yes','no'],
}

## 2. Đọc dữ liệu
File `bank-full.csv` nằm cùng thư mục với notebook, dùng dấu phân cách `;`.

In [2]:
DATA_PATH = "bank-full.csv"
df = pd.read_csv(DATA_PATH, sep=";")
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 3. Khám phá nhanh
Xem kiểu dữ liệu, giá trị thiếu, thống kê mô tả và phân bố biến mục tiêu trước khi làm sạch.

In [3]:
df.info()
print("Ô thiếu:", int(df.isna().sum().sum()), "| Dòng trùng:", int(df.duplicated().sum()))
print(df['y'].value_counts(normalize=True).round(3))
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
Ô thiếu: 0 | Dòng trùng: 0
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


## 4. Kiểm tra ràng buộc nghiệp vụ
Hàm `validate` trả về hai nhóm mask các dòng vi phạm:
- **checks:** khoảng giá trị hợp lệ (tuổi, thời lượng, số lần liên hệ, ngày) và nhãn phân loại hợp lệ.
- **consistency:** nhất quán giữa các cột `pdays`, `previous`, `poutcome`.

In [4]:
def validate(df):
    m = df['month'].map(MONTH_NUM)
    max_day = m.map(DAYS_IN_MONTH)
    checks = {
        'age ngoài [18, 100]':            ~df['age'].between(18, 100),
        'duration <= 0':                  df['duration'] <= 0,
        'campaign < 1':                   df['campaign'] < 1,
        'previous < 0':                   df['previous'] < 0,
        'pdays < -1':                     df['pdays'] < -1,
        'day ngoài [1, 31]':              ~df['day'].between(1, 31),
        'ngày không tồn tại trong tháng': df['day'] > max_day,
        **{f'{c} sai nhãn': ~df[c].isin(v) for c, v in ALLOWED.items()},
    }
    consistency = {
        'pdays=-1 nhưng previous>0':            (df['pdays'] == -1) & (df['previous'] > 0),
        'pdays>=0 nhưng previous=0':            (df['pdays'] >= 0) & (df['previous'] == 0),
        'previous=0 nhưng poutcome != unknown': (df['previous'] == 0) & (df['poutcome'] != 'unknown'),
        'poutcome=success nhưng previous=0':    (df['poutcome'] == 'success') & (df['previous'] == 0),
    }
    return checks, consistency

checks, consistency = validate(df)
report = pd.Series({**{k: int(v.sum()) for k, v in checks.items()},
                    **{k: int(v.sum()) for k, v in consistency.items()}})
print(report[report > 0] if (report > 0).any() else "Không có vi phạm")

duration <= 0    3
dtype: int64


## 5. Data Cleaning
1. Chuẩn hóa chuỗi (cắt khoảng trắng, chữ thường)
2. Loại dòng không hợp lệ và dòng trùng lặp
3. Loại dòng mâu thuẫn giữa các cột `pdays` / `previous` / `poutcome`
4. Logic-based imputation (`poutcome`, `education`, `job`; `contact = unknown` giữ nguyên)
5. Xử lý ngoại lai bằng IQR (giới hạn giá trị, không xóa dòng)
6. Data transformation (mã hóa biến mục tiêu `y` thành 1/0)
7. Ép kiểu chuẩn

In [5]:
def _group_mode_impute(df, target, by):
    # Thay 'unknown' của cột target bằng mode của nhóm `by` (tính trên các dòng đã biết)
    mask = df[target] == 'unknown'
    known = df.loc[~mask]
    mode_map = (known.groupby(by, observed=True)[target]
                     .agg(lambda s: s.mode().iat[0]).rename('_mode'))
    filled = df.loc[mask, by].join(mode_map, on=by)['_mode']
    df.loc[mask, target] = filled.fillna('unknown').values
    return df


def handle_outliers_iqr(df, column, k=1.5, floor=None):
    # Giới hạn ngoại lai theo IQR. floor: chỉ tính trên các giá trị >= floor (bỏ qua mã đặc biệt như -1, 0)
    s = df[column].astype(float)
    mask = (s >= floor) if floor is not None else s.notna()
    q1, q3 = s[mask].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    n_out = int(((s[mask] < low) | (s[mask] > high)).sum())
    df[column] = s
    df.loc[mask, column] = s[mask].clip(low, high)
    return df, n_out


def clean_bank_dataset(df_input, drop_inconsistent=True, cap_outliers=True, iqr_k=1.5, verbose=True):
    df = df_input.copy()
    log = {'dòng ban đầu': len(df)}

    # --- BƯỚC 1: Chuẩn hóa chuỗi ---
    for c in df.select_dtypes(include=['object', 'string']).columns:
        df[c] = df[c].str.strip().str.lower()

    # --- BƯỚC 2: Loại dòng không hợp lệ và dòng trùng lặp ---
    checks, _ = validate(df)
    invalid = pd.concat(checks.values(), axis=1).any(axis=1)
    log['loại do giá trị không hợp lệ'] = int(invalid.sum())
    df = df[~invalid].copy()

    n = len(df)
    df = df.drop_duplicates()
    log['loại do trùng lặp'] = n - len(df)

    # --- BƯỚC 3: Nhất quán giữa các cột ---
    _, consistency = validate(df)
    bad = pd.concat(consistency.values(), axis=1).any(axis=1)
    log['dòng mâu thuẫn pdays/previous/poutcome'] = int(bad.sum())
    if drop_inconsistent:
        df = df[~bad].copy()

    # --- BƯỚC 4: Logic-based imputation ---
    never_contacted = (df['previous'] == 0) & (df['pdays'] == -1)
    df.loc[never_contacted & (df['poutcome'] == 'unknown'), 'poutcome'] = 'nonexistent'

    df['age_bin'] = pd.cut(df['age'], [0, 30, 45, 60, 120],
                           labels=['<=30', '31-45', '46-60', '>60']).astype(str)   # cột tạm, xóa sau khi điền
    before = {c: int((df[c] == 'unknown').sum()) for c in ['education', 'job']}
    df = _group_mode_impute(df, 'education', ['job'])
    df = _group_mode_impute(df, 'job', ['education', 'age_bin'])
    df = _group_mode_impute(df, 'education', ['job'])
    log['unknown được điền'] = {c: before[c] - int((df[c] == 'unknown').sum()) for c in before}
    df = df.drop(columns='age_bin')

    # --- BƯỚC 5: Xử lý ngoại lai (IQR) ---
    if cap_outliers:
        capped = {}
        for c in ['balance', 'duration', 'campaign']:
            df, capped[c] = handle_outliers_iqr(df, c, k=iqr_k)
        # pdays, previous: phần lớn là mã "chưa liên hệ" (-1 / 0) nên chỉ tính trên giá trị thực
        for c, floor in [('pdays', 0), ('previous', 1)]:
            df, capped[c] = handle_outliers_iqr(df, c, k=iqr_k, floor=floor)
        log['ngoại lai bị giới hạn'] = capped

    # --- BƯỚC 6: Data transformation ---
    df['y'] = df['y'].map({'yes': 1, 'no': 0})

    # --- BƯỚC 7: Ép kiểu chuẩn ---
    for c in ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']:
        df[c] = df[c].astype('category')
    for c in ['age', 'day']:
        df[c] = df[c].astype('int64')

    df = df.reset_index(drop=True)
    log['dòng sau làm sạch'] = len(df)
    if verbose:
        for k, v in log.items():
            print(f"{k}: {v}")
    return df

## 6. Chạy làm sạch


In [6]:
df_cleaned = clean_bank_dataset(df)
df_cleaned.head()

dòng ban đầu: 45211
loại do giá trị không hợp lệ: 3
loại do trùng lặp: 0
dòng mâu thuẫn pdays/previous/poutcome: 0
unknown được điền: {'education': 1857, 'job': 288}
ngoại lai bị giới hạn: {'balance': 4728, 'duration': 3235, 'campaign': 3063, 'pdays': 49, 'previous': 453}
dòng sau làm sạch: 45208


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,261.0,1.0,-1.0,0.0,nonexistent,0
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5,may,151.0,1.0,-1.0,0.0,nonexistent,0
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5,may,76.0,1.0,-1.0,0.0,nonexistent,0
3,47,blue-collar,married,secondary,no,1506.0,yes,no,unknown,5,may,92.0,1.0,-1.0,0.0,nonexistent,0
4,33,blue-collar,single,secondary,no,1.0,no,no,unknown,5,may,198.0,1.0,-1.0,0.0,nonexistent,0


## 7. Kiểm tra sau làm sạch
Xác nhận không còn ô thiếu, không còn vi phạm ràng buộc, và xem lại phân bố các biến.

In [7]:
print(df_cleaned.shape, "| ô thiếu:", int(df_cleaned.isna().sum().sum()))
print("unknown còn lại:", {c: int((df_cleaned[c] == 'unknown').sum()) for c in ['job', 'education', 'contact', 'poutcome']})
print(df_cleaned['poutcome'].value_counts())
df_cleaned.describe().T

(45208, 17) | ô thiếu: 0
unknown còn lại: {'job': 0, 'education': 0, 'contact': 13019, 'poutcome': 5}
poutcome
nonexistent    36951
failure         4901
other           1840
success         1511
unknown            5
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
age,45208.0,40.936294,10.618823,18.0,33.0,39.0,48.0,95.0
balance,45208.0,933.681008,1176.735545,-1962.0,72.0,448.0,1428.0,3462.0
day,45208.0,15.806052,8.322417,1.0,8.0,16.0,21.0,31.0
duration,45208.0,234.954897,176.745434,1.0,103.0,180.0,319.0,643.0
campaign,45208.0,2.392121,1.600055,1.0,1.0,2.0,3.0,6.0
pdays,45208.0,40.060100,99.201640,-1.0,-1.0,-1.0,-1.0,618.0
previous,45208.0,0.520715,1.435200,0.0,0.0,0.0,0.0,8.5
y,45208.0,0.116993,0.321415,0.0,0.0,0.0,0.0,1.0


## 8. Xuất file
Lưu dữ liệu đã làm sạch để dùng cho các bước EDA và huấn luyện mô hình.

Lưu ý: `duration` chỉ biết sau khi cuộc gọi kết thúc, nên nếu mục tiêu là dự đoán trước cuộc gọi thì cân nhắc loại biến này khi huấn luyện mô hình.

In [8]:
df_cleaned.to_csv('bank_data_cleaned.csv', index=False)
print("Đã lưu bank_data_cleaned.csv")

Đã lưu bank_data_cleaned.csv
